# 01 - Senior et al. 2020: AlphaFold CASP13 Reproduction

This notebook is a **from-scratch PyTorch reproduction** of the first AlphaFold paper, not a wrapper around DeepMind's CASP13 code or weights.

The Senior et al. system can be decomposed into four reproducible ideas:

1. Build leakage-controlled sequence/template features for CASP13 targets.
2. Train a residual network that predicts inter-residue distance distributions, torsions, and auxiliary structure features.
3. Convert predicted distance probabilities into a differentiable potential of mean force.
4. Optimize 3D coordinates under that potential and score them against CASP13 structures.

The first faithful target is "same benchmark, same information boundary." The improvement target is "beat their benchmark with clearly labeled enhanced experiments."

## Data and model layout

All input data and trained checkpoints are ours:

- `data/senior_2020/raw`: CASP13 FASTA, ground-truth structures, dated PDB/CATH snapshots, and MSA database snapshots.
- `data/senior_2020/features`: tensorized MSA/template/contact labels.
- `models/senior_2020/checkpoints`: our PyTorch checkpoints.
- `results/senior_2020`: distograms, optimized PDBs, and score tables.

No official AlphaFold API, Docker runner, model parameters, or inference script is used.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import shlex
import subprocess
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / "results"
RUNS_ROOT = PROJECT_ROOT / "runs"

for path in [DATA_ROOT, MODEL_ROOT, RESULTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def latest_environment_report() -> dict:
    report_dir = DATA_ROOT / "environment_reports"
    reports = sorted(report_dir.glob("environment_report_*_utc.json"))
    if not reports:
        return {}
    return json.loads(reports[-1].read_text(encoding="utf-8"))

ENV_REPORT = latest_environment_report()

def seed_everything(seed: int = 7) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def device() -> torch.device:
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def write_text(path: Path, text: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def run(cmd: list[str], *, cwd: Path | None = None, timeout: int = 30, dry_run: bool = True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if dry_run:
        print("DRY_RUN=True, command was not executed.")
        return None
    completed = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed

def gpu_summary() -> str:
    devices = ENV_REPORT.get("torch", {}).get("devices", [])
    if devices:
        first = devices[0]
        return f"{first.get('name')} / {first.get('total_memory_gb')} GB / cc {first.get('major')}.{first.get('minor')}"
    return "No saved GPU report found."

seed_everything(7)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device()}")
print(f"Saved cluster GPU: {gpu_summary()}")

Project root: /workspace
Device: cuda:0
Saved cluster GPU: No saved GPU report found.


In [2]:
PAPER = "senior_2020"
DATA_DIR = DATA_ROOT / PAPER
MODEL_DIR = MODEL_ROOT / PAPER
RESULT_DIR = RESULTS_ROOT / PAPER
RUN_DIR = RUNS_ROOT / PAPER

paths = {
    "raw": DATA_DIR / "raw",
    "features": DATA_DIR / "features",
    "labels": DATA_DIR / "labels",
    "checkpoints": MODEL_DIR / "checkpoints",
    "distograms": RESULT_DIR / "distograms",
    "structures": RESULT_DIR / "structures",
    "metrics": RESULT_DIR / "metrics",
    "slurm": RUN_DIR / "slurm",
}
for p in paths.values():
    p.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

{
  "raw": "/workspace/data/senior_2020/raw",
  "features": "/workspace/data/senior_2020/features",
  "labels": "/workspace/data/senior_2020/labels",
  "checkpoints": "/workspace/models/senior_2020/checkpoints",
  "distograms": "/workspace/results/senior_2020/distograms",
  "structures": "/workspace/results/senior_2020/structures",
  "metrics": "/workspace/results/senior_2020/metrics",
  "slurm": "/workspace/runs/senior_2020/slurm"
}


## Step 1 - Benchmark contract

The contract below is the guardrail against accidental leakage. A faithful Senior reproduction should not train on structures that post-date CASP13 and should not use sequence/template databases newer than the paper-era versions. Later, enhanced experiments may use modern databases, but those rows must be marked `faithful=False`.

Scientifically, this step defines the counterfactual question: "What could the method have known at CASP13 time?" Protein-structure predictors can silently improve just by seeing homologous future structures or larger modern sequence databases, so the benchmark boundary is part of the experiment, not bookkeeping.

Computationally, we serialize the boundary as JSON so every later feature file, checkpoint, and score table can point back to the same data contract. Mathematically, it fixes the training and evaluation distributions: training samples are drawn from structures and alignments available before the cutoff, while CASP13 targets remain held out. Without that separation, a high TM-score would not estimate generalization.

In [3]:
contract = {
    "paper": "Senior et al. 2020",
    "benchmark": "CASP13 free modelling domains",
    "template_cutoff": "2018-03-15",
    "cath_cutoff": "2018-03-16",
    "uniclust30_version": "2017-10",
    "nr_snapshot": "2017-12-15",
    "implementation": "own_pytorch",
    "uses_official_weights_or_api": False,
}
write_text(DATA_DIR / "benchmark_contract.json", json.dumps(contract, indent=2))
print(json.dumps(contract, indent=2))

{
  "paper": "Senior et al. 2020",
  "benchmark": "CASP13 free modelling domains",
  "template_cutoff": "2018-03-15",
  "cath_cutoff": "2018-03-16",
  "uniclust30_version": "2017-10",
  "nr_snapshot": "2017-12-15",
  "implementation": "own_pytorch",
  "uses_official_weights_or_api": false
}


## Step 2 - Feature tensors

Senior-style distance prediction is naturally pairwise. We represent each protein as:

- `msa_profile`: `[L, A]`, amino-acid frequencies from the MSA.
- `msa_covariance`: `[L, L, C]`, cheap co-evolution/covariance summaries.
- `template_pair`: `[L, L, T]`, template-derived distances/masks if allowed by cutoff.
- `dist_bin`: `[L, L]`, supervised C-beta/C-alpha distance bin label.

The dataset class below is intentionally file-format simple. Feature builders can write `.npz` files from HHblits/JackHMMER/template tools later, while the model and training code stay stable.

Scientifically, the MSA summarizes evolutionary pressure: residues that mutate together often contact or constrain each other in 3D. Templates encode the separate hypothesis that homologous folds provide partial geometry when they are legitimately available before the cutoff.

Computationally, we turn variable biological evidence into dense tensors. The central tensor is `[L, L, C]`, where each entry describes a residue pair. Mathematically, the supervised label is a binned distance random variable `Y_ij`; the model will learn `p(Y_ij = b | MSA, templates)` for every residue pair `(i, j)`.

In [4]:
class SeniorFeatureDataset(Dataset):
    def __init__(self, feature_dir: Path, synthetic_if_empty: bool = True, n_synthetic: int = 8, length: int = 96, bins: int = 32):
        self.files = sorted(feature_dir.glob("*.npz"))
        self.synthetic_if_empty = synthetic_if_empty
        self.n_synthetic = n_synthetic
        self.length = length
        self.bins = bins

    def __len__(self):
        return len(self.files) if self.files else (self.n_synthetic if self.synthetic_if_empty else 0)

    def __getitem__(self, idx):
        if self.files:
            arr = np.load(self.files[idx])
            return {k: torch.as_tensor(arr[k]) for k in arr.files}

        L, B = self.length, self.bins
        msa_profile = F.one_hot(torch.randint(0, 21, (L,)), 21).float()
        msa_covariance = torch.randn(L, L, 16) * 0.1
        template_pair = torch.zeros(L, L, 8)
        pair = torch.cat([
            msa_profile[:, None, :].expand(L, L, 21),
            msa_profile[None, :, :].expand(L, L, 21),
            msa_covariance,
            template_pair,
        ], dim=-1)
        dist_bin = torch.randint(0, B, (L, L))
        dist_bin = torch.triu(dist_bin, diagonal=1) + torch.triu(dist_bin, diagonal=1).T
        return {"pair": pair, "dist_bin": dist_bin}

dataset = SeniorFeatureDataset(paths["features"])
batch = dataset[0]
print({k: tuple(v.shape) for k, v in batch.items()})

{'pair': (96, 96, 66), 'dist_bin': (96, 96)}


## Step 3 - Our Senior-style distogram network

The original system used deep residual networks over pair features. We keep that inductive bias: 2D convolutions operate on the residue-pair matrix, preserving local patterns in sequence separation and pairwise geometry. The network predicts distance bins for every residue pair.

Scientifically, a distogram is richer than a contact map because it preserves approximate geometry instead of only saying "near" or "far." That matters because many folds can satisfy the same binary contacts, but far fewer satisfy a full collection of distance distributions.

Computationally, the model is a 2D residual CNN over the residue-pair image. Residual connections let us stack many transformations without destroying gradient flow. Mathematically, the final head produces logits `z_ijb`, and `softmax_b(z_ijb)` is our estimate of the distance-bin distribution for pair `(i, j)`.

In [5]:
class Residual2DBlock(nn.Module):
    def __init__(self, channels: int, dilation: int = 1):
        super().__init__()
        pad = dilation
        self.net = nn.Sequential(
            nn.GroupNorm(8, channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=pad, dilation=dilation),
            nn.GroupNorm(8, channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.net(x)


class SeniorDistogramNet(nn.Module):
    def __init__(self, pair_dim: int = 66, hidden: int = 128, blocks: int = 16, bins: int = 32):
        super().__init__()
        self.stem = nn.Conv2d(pair_dim, hidden, 1)
        self.blocks = nn.ModuleList([Residual2DBlock(hidden, dilation=1 + (i % 4)) for i in range(blocks)])
        self.dist_head = nn.Conv2d(hidden, bins, 1)

    def forward(self, pair):
        # pair: [B, L, L, C]
        x = pair.permute(0, 3, 1, 2).contiguous()
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        logits = self.dist_head(x).permute(0, 2, 3, 1).contiguous()
        logits = 0.5 * (logits + logits.transpose(1, 2))
        return {"distogram_logits": logits}

model = SeniorDistogramNet().to(device())
with torch.no_grad():
    out = model(batch["pair"][None].to(device()))
print(tuple(out["distogram_logits"].shape))

(1, 96, 96, 32)


## Step 4 - Training objective

The distogram target is a categorical distance distribution. We train with cross-entropy over residue pairs, excluding the diagonal and optionally downweighting pairs close in sequence if we want to emphasize long-range structure. This is the learning step that turns MSA/template evidence into geometric constraints.

Scientifically, the network is asked to learn statistical regularities between evolutionary features and physical distances: co-evolving residues, conserved motifs, and template hints should increase probability mass on compatible distance bins.

Computationally, each protein contributes roughly `O(L^2)` residue-pair examples, so masking and batching matter on GPU. Mathematically, the loss is the negative log-likelihood `-sum_ij log p_theta(y_ij | x)` over valid residue pairs. Minimizing this trains the model to assign calibrated probability mass to the observed structure-derived bins.

In [6]:
def senior_loss(outputs, target_bins):
    logits = outputs["distogram_logits"]
    B, L, _, bins = logits.shape
    mask = torch.triu(torch.ones(L, L, device=logits.device, dtype=torch.bool), diagonal=1)
    loss = F.cross_entropy(logits[:, mask, :].reshape(-1, bins), target_bins[:, mask].reshape(-1))
    return loss

loader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
model.train()
for step, item in enumerate(loader):
    item = {k: v.to(device()) for k, v in item.items()}
    optimizer.zero_grad(set_to_none=True)
    loss = senior_loss(model(item["pair"]), item["dist_bin"].long())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    print(f"step={step} loss={float(loss.detach().cpu()):.4f}")
    if step >= 2:
        break

step=0 loss=3.8327
step=1 loss=5.1782
step=2 loss=4.8997


## Step 5 - Potential of mean force and coordinate optimization

The network predicts probabilities, not coordinates. We reproduce the paper's core move: transform log-probabilities into a coordinate potential and optimize a chain. This code is deliberately PyTorch-native so later improvements can backpropagate through parts of the pipeline if useful.

Scientifically, this converts learned geometric preferences into an actual fold. The model says which distances are plausible; coordinate optimization asks for a 3D chain whose pairwise distances satisfy those preferences while remaining protein-like.

Computationally, we optimize coordinates directly with automatic differentiation. Mathematically, the potential is an energy `E(x) = -sum_ij log p_ij(bin(||x_i - x_j||))` with a reference correction and chain regularizers. Gradient descent changes coordinates in the direction that lowers this learned energy landscape.

In [7]:
def distogram_energy(coords, bin_edges, pair_log_probs, reference_log_probs):
    distances = torch.cdist(coords, coords).clamp_min(1e-6)
    bin_idx = torch.bucketize(distances, bin_edges[1:-1])
    observed = pair_log_probs.gather(-1, bin_idx[..., None]).squeeze(-1)
    reference = reference_log_probs[bin_idx]
    mask = torch.triu(torch.ones_like(distances, dtype=torch.bool), diagonal=2)
    return -(observed - reference)[mask].mean()

def optimize_trace(pair_logits, steps: int = 300, lr: float = 0.05):
    L, _, bins = pair_logits.shape
    dev = pair_logits.device
    bin_edges = torch.linspace(2.0, 22.0, bins + 1, device=dev)
    pair_log_probs = pair_logits.log_softmax(dim=-1)
    reference = torch.full((bins,), -math.log(bins), device=dev)
    coords = torch.randn(L, 3, device=dev, requires_grad=True)
    opt = torch.optim.Adam([coords], lr=lr)
    history = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        energy = distogram_energy(coords, bin_edges, pair_log_probs, reference)
        ca = (coords[1:] - coords[:-1]).norm(dim=-1)
        chain_loss = ((ca - 3.8) ** 2).mean()
        loss = energy + 0.05 * chain_loss
        loss.backward()
        opt.step()
        if step % 50 == 0 or step == steps - 1:
            history.append({"step": step, "loss": float(loss.detach().cpu()), "energy": float(energy.detach().cpu())})
    return coords.detach(), history

model.eval()
with torch.no_grad():
    logits = model(batch["pair"][None].to(device()))["distogram_logits"][0]
coords, hist = optimize_trace(logits, steps=20)
print(hist[-1], tuple(coords.shape))

{'step': 19, 'loss': 0.7935764193534851, 'energy': 0.786684513092041} (96, 3)


## Step 6 - Scoring and improvement loop

Final scoring should use external structural metrics such as TM-score/US-align and CASP-style GDT. The notebook writes score schemas and experiment registries rather than hard-coding a single binary path.

Scientifically, score choice determines what "reproduction" means. TM-score and GDT measure global fold accuracy, while later additions such as lDDT or violation counts can expose local geometry failures that a global metric may hide.

Computationally, we separate metric schemas from metric binaries because clusters differ in installed tools. Mathematically, each experiment row is a paired comparison between predicted coordinates `X_hat` and reference coordinates `X`, after alignment or distance-based matching. The registry also prevents enhanced experiments from being mistaken for faithful reproduction.

In [8]:
score_schema = {
    "target_id": "T0986s2",
    "prediction_path": "results/senior_2020/structures/T0986s2/model_0.pdb",
    "truth_path": "data/senior_2020/raw/casp13_targets/T0986s2.pdb",
    "tm_score": None,
    "gdt_ts": None,
    "faithful": True,
    "implementation": "own_pytorch",
}
experiments = [
    {"name": "senior_faithful_resnet_distogram", "faithful": True, "change": "paper-era data, own residual distogram model"},
    {"name": "senior_modern_msa", "faithful": False, "change": "new sequence databases"},
    {"name": "senior_af2_prior_potential", "faithful": False, "change": "AF2-like prior in coordinate optimizer"},
]
write_text(paths["metrics"] / "score_schema.json", json.dumps(score_schema, indent=2))
write_text(RUN_DIR / "experiment_registry.json", json.dumps(experiments, indent=2))
print(json.dumps(experiments, indent=2))

[
  {
    "name": "senior_faithful_resnet_distogram",
    "faithful": true,
    "change": "paper-era data, own residual distogram model"
  },
  {
    "name": "senior_modern_msa",
    "faithful": false,
    "change": "new sequence databases"
  },
  {
    "name": "senior_af2_prior_potential",
    "faithful": false,
    "change": "AF2-like prior in coordinate optimizer"
  }
]


## Cluster execution template

The notebooks are deliberately importable and runnable from `papermill`, `jupyter nbconvert --execute`, or ordinary notebook execution. For long runs on the cluster, the code writes a small SLURM script that executes this notebook with parameters rather than keeping GPU time tied to the browser session.

In [9]:
slurm = paths["slurm"] / "train_senior_distogram.sbatch"
slurm.write_text(f"""#!/usr/bin/env bash
#SBATCH --job-name=senior-dist
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output={paths['slurm']}/%x-%j.out
set -euo pipefail
cd "{PROJECT_ROOT}"
jupyter nbconvert --to notebook --execute 01_Senior_2020_AlphaFold_CASP13_Reproduction.ipynb --output results/senior_2020/executed.ipynb
""", encoding="utf-8")
print(slurm.read_text(encoding="utf-8"))

#!/usr/bin/env bash
#SBATCH --job-name=senior-dist
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output=/workspace/runs/senior_2020/slurm/%x-%j.out
set -euo pipefail
cd "/workspace"
jupyter nbconvert --to notebook --execute 01_Senior_2020_AlphaFold_CASP13_Reproduction.ipynb --output results/senior_2020/executed.ipynb

